In [ ]:
#import
import sys
sys.path.append("..")
from topofinger import config
import numpy as np
import matplotlib.pyplot as plt

import sys
from topofinger.viz import plot_identifiability
from topofinger.utils import threshold_matrix

%load_ext autoreload
%autoreload 2

#paths
sup_folder = config.HCP_1200 + "/NoGSR_LowPassOnly/"
scaffolds_folder = sup_folder + "scaffolds/whole/"
fc_folder = sup_folder + "fc/whole/"
figure_folder = config.FIGURES_RAW + "/final/fig2/"

# Create figure folder if it doesn't exist
import os
os.makedirs(figure_folder, exist_ok=True)

import seaborn as sb
import pandas as pd

from pingouin import compute_effsize
from statannotations.Annotator import Annotator

import matplotlib.transforms as transforms
from matplotlib.ticker import FormatStrFormatter

import random

from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate

def load_scaffold(path, n_regions_fc=300):
    """Load a scaffold file, handling axis order and region count differences.
    
    1200subjects: shape (300, 300, n_win), fortran-order int16
    100unrelated: shape (n_win, 300, 300), fortran-order float64
    """
    scs = np.load(path)
    # If first two dims are equal and differ from last → (regions, regions, n_windows)
    if scs.ndim == 3 and scs.shape[0] == scs.shape[1] and scs.shape[0] != scs.shape[2]:
        scs = scs.transpose(2, 0, 1)  # → (n_windows, regions, regions)
    # Trim to match FC region count if needed (drop subcortical Tian regions)
    if scs.shape[1] > n_regions_fc:
        scs = scs[:, :n_regions_fc, :n_regions_fc]
    return scs.astype(np.float64)

def get_scaffold_filename(fc_filename, scaffold_dir):
    """Resolve scaffold filename from FC filename, handling naming conventions.
    
    1200subjects: {sub_id}_s01.npy
    100unrelated: {sub_id} (no extension, no suffix)
    """
    sub_id = fc_filename.split('_')[0]
    # Try 1200subjects convention first
    candidate = os.path.join(scaffold_dir, f'{sub_id}_s01.npy')
    if os.path.isfile(candidate):
        return candidate
    # Try 100unrelated convention (no extension)
    candidate = os.path.join(scaffold_dir, sub_id)
    if os.path.isfile(candidate):
        return candidate
    return None

def get_valid_subs_for_width(width, data_dir):
    """Get subjects with matching FC+scaffold files and consistent window counts."""
    fc_dir = data_dir + f'fc/{width}/'
    sc_dir = data_dir + f'scaffolds/windows_cortex/{width}/freq/'
    
    # First pass: find all subjects with both files and their window counts
    sub_wins = {}
    for sub in sorted(os.listdir(fc_dir)):
        sc_path = get_scaffold_filename(sub, sc_dir)
        if sc_path is None:
            continue
        fcs = np.load(data_dir + f'fc/{width}/{sub}', mmap_mode='r')
        sub_wins[sub] = fcs.shape[0]
    
    # Keep only subjects with the most common window count
    from collections import Counter
    if not sub_wins:
        return [], 0
    win_counts = Counter(sub_wins.values())
    modal_n_win = win_counts.most_common(1)[0][0]
    valid = [sub for sub, n in sub_wins.items() if n == modal_n_win]
    return valid, modal_n_win


## Panel A (currently with 1200 subjects data)

In [ ]:
# LOAD CORRELATION FINGERPRINTING DATA ##

sc = np.load(scaffolds_folder + 'scaffold_correlations_whole_cortex.npy')
fc100 = np.load(fc_folder + 'fc_correlations_whole_th100_cortex.npy')
fc25 = np.load(fc_folder + 'fc_correlations_whole_th25_cortex.npy')
fc1 = np.load(fc_folder + 'fc_correlations_whole_th1_cortex.npy')

In [ ]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))
nsub_to_plot = 100

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65

# Create a shared colormap with the same vmin and vmax
im1 = ax1.imshow(fc100[:nsub_to_plot,:nsub_to_plot], cmap=cmap, vmin=vmin, vmax=vmax)
ax1.set_title("FC full", fontsize=20)
ax1.set_xticks([])
ax1.set_yticks([])
success_rate1, success_rate2 = compute_success_rate(fc100)
ax1.text(0.5, -0.07, f"Retest", transform=ax1.transAxes, ha='center', fontsize=18)
ax1.text(0.5, -0.2, f'\n\nSuccess Rate = {(success_rate1+success_rate2)/2:.1f}%', transform=ax1.transAxes, ha='center', fontsize=18)

im2 = ax2.imshow(fc25[:nsub_to_plot,:nsub_to_plot], cmap=cmap, vmin=vmin, vmax=vmax)
ax2.set_title("FC, th=25%", fontsize=20)
ax2.set_xticks([])
ax2.set_yticks([])
success_rate1, success_rate2 = compute_success_rate(fc25)
ax2.text(0.5, -0.07, f"Retest", transform=ax2.transAxes, ha='center', fontsize=18)
ax2.text(0.5, -0.2, f'\n\nSuccess Rate = {(success_rate1+success_rate2)/2:.1f}%', transform=ax2.transAxes, ha='center', fontsize=18)

im3 = ax3.imshow(fc1[:nsub_to_plot,:nsub_to_plot], cmap=cmap, vmin=vmin, vmax=vmax)
ax3.set_title("FC, th=1%", fontsize=20)
ax3.set_xticks([])
ax3.set_yticks([])
success_rate1, success_rate2 = compute_success_rate(fc1)
ax3.text(0.5, -0.07, f"Retest", transform=ax3.transAxes, ha='center', fontsize=18)
ax3.text(0.5, -0.2, f'\n\nSuccess Rate = {(success_rate1+success_rate2)/2:.1f}%', transform=ax3.transAxes, ha='center', fontsize=18)

im4 = ax4.imshow(sc[:nsub_to_plot,:nsub_to_plot], cmap=cmap, vmin=vmin, vmax=vmax)
ax4.set_title("Scaffolds", fontsize=20)
ax4.set_xticks([])
ax4.set_yticks([])
success_rate1, success_rate2 = compute_success_rate(sc)
ax4.text(0.5, -0.07, f"Retest", transform=ax4.transAxes, ha='center', fontsize=18)
ax4.text(0.5, -0.2, f' \n\nSuccess Rate = {(success_rate1+success_rate2)/2:.1f}%', transform=ax4.transAxes, ha='center', fontsize=18, weight='bold')

# Add "Retest" label under each plot
ax1.text(-0.07, 0.5, 'Test', va='center', rotation=90, fontsize=18, transform=ax1.transAxes)

# Create a single colorbar for all plots with position on the right side
plt.subplots_adjust(right=0.9, wspace=0.05, bottom=0.15)  # Adjusted bottom to make room for the shared x-label
cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])  # [x0, y0, width, height]
cbar = fig.colorbar(im1, cax=cbar_ax)
cbar.ax.tick_params(labelsize=16)

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2A.png'), dpi=300, bbox_inches='tight')
plt.show()


## Panel B (currently with 1200 subjects data)

In [ ]:
# Extract intra-subject (diagonal) and inter-subject (off-diagonal) correlations for all conditions
intra_subject_fc100 = np.diag(fc100)
inter_subject_fc100 = fc100[~np.eye(fc100.shape[0], dtype=bool)]

intra_subject_fc25 = np.diag(fc25)
inter_subject_fc25 = fc25[~np.eye(fc25.shape[0], dtype=bool)]

intra_subject_fc1 = np.diag(fc1)
inter_subject_fc1 = fc1[~np.eye(fc1.shape[0], dtype=bool)]

intra_subject_sc = np.diag(sc)
inter_subject_sc = sc[~np.eye(sc.shape[0], dtype=bool)]


# Create dataframes for each condition
data_fc100 = {
    'distribution': (
        ['intra_subject_fc100'] * len(intra_subject_fc100)
        + ['inter_subject_fc100'] * len(inter_subject_fc100)
    ),
    'value': np.concatenate([intra_subject_fc100, inter_subject_fc100])
}

data_fc25 = {
    'distribution': (
        ['intra_subject_fc25'] * len(intra_subject_fc25)
        + ['inter_subject_fc25'] * len(inter_subject_fc25)
    ),
    'value': np.concatenate([intra_subject_fc25, inter_subject_fc25])
}

data_fc1 = {
    'distribution': (
        ['intra_subject_fc1'] * len(intra_subject_fc1)
        + ['inter_subject_fc1'] * len(inter_subject_fc1)
    ),
    'value': np.concatenate([intra_subject_fc1, inter_subject_fc1])
}

data_sc = {
    'distribution': (
        ['intra_subject_sc'] * len(intra_subject_sc)
        + ['inter_subject_sc'] * len(inter_subject_sc)
    ),
    'value': np.concatenate([intra_subject_sc, inter_subject_sc])
}

# Convert to pandas dataframes
df_fc100 = pd.DataFrame(data_fc100)
df_fc25 = pd.DataFrame(data_fc25)
df_fc1 = pd.DataFrame(data_fc1)
df_sc = pd.DataFrame(data_sc)

# Compute effect sizes for each condition
es_fc100 = compute_effsize(df_fc100.loc[df_fc100['distribution'] == 'intra_subject_fc100', 'value'], 
                         df_fc100.loc[df_fc100['distribution'] == 'inter_subject_fc100', 'value'], paired=False)
es_fc25 = compute_effsize(df_fc25.loc[df_fc25['distribution'] == 'intra_subject_fc25', 'value'], 
                        df_fc25.loc[df_fc25['distribution'] == 'inter_subject_fc25', 'value'], paired=False)
es_fc1 = compute_effsize(df_fc1.loc[df_fc1['distribution'] == 'intra_subject_fc1', 'value'], 
                       df_fc1.loc[df_fc1['distribution'] == 'inter_subject_fc1', 'value'], paired=False)
es_sc = compute_effsize(df_sc.loc[df_sc['distribution'] == 'intra_subject_sc', 'value'], 
                      df_sc.loc[df_sc['distribution'] == 'inter_subject_sc', 'value'], paired=False)

In [ ]:
# Color palettes for the plots
blue_palette = sb.color_palette("Blues", n_colors=2)[::-1]  # For FC plots
orange_palette = sb.color_palette("Oranges", n_colors=2)[::-1]  # For scaffold plot
p_size = 7
# Create a 1x4 subplot layout like Panel A
fig, axes = plt.subplots(1, 4, figsize=(21, 5.5), sharey=True)

# FC100 plot
sb.violinplot(x="distribution", y="value", data=df_fc100, ax=axes[0], split=True, inner=None, hue="distribution", palette=blue_palette, legend=False)
sb.stripplot(x="distribution", y="value", data=df_fc100, ax=axes[0], alpha=0.6, size=p_size, jitter=0.002, hue="distribution", palette=blue_palette, legend=False)

# FC25 plot
sb.violinplot(x="distribution", y="value", data=df_fc25, ax=axes[1], split=True, inner=None, hue='distribution', palette=blue_palette, legend=False)
sb.stripplot(x="distribution", y="value", data=df_fc25, ax=axes[1], alpha=0.6, size=p_size, jitter=0.002, hue='distribution', palette=blue_palette, legend=False)

# FC1 plot
sb.violinplot(x="distribution", y="value", data=df_fc1, ax=axes[2], split=True, inner=None, hue='distribution', palette=blue_palette, legend=False)
sb.stripplot(x="distribution", y="value", data=df_fc1, ax=axes[2], alpha=0.6, size=p_size, jitter=0.002, hue='distribution', palette=blue_palette, legend=False)

# Scaffolds plot
sb.violinplot(x="distribution", y="value", data=df_sc, ax=axes[3], split=True, inner=None, hue='distribution', palette=orange_palette, legend=False)
sb.stripplot(x="distribution", y="value", data=df_sc, ax=axes[3], alpha=0.6, size=p_size, jitter=0.002, hue='distribution', palette=orange_palette, legend=False)

# Apply offset for better visualization of points
offset = transforms.ScaledTranslation(0.87, 0, fig.dpi_scale_trans)
for ax in axes:
    for collection in ax.collections[-2:-1]:
        collection.set_transform(collection.get_transform() + offset)
    for collection in ax.collections[-1:]:
        collection.set_transform(collection.get_transform() - offset)
        collection.set_transform(collection.get_transform() - offset)
        collection.set_transform(collection.get_transform() - offset)

# Set titles and labels
axes[0].set_title("FC full", fontsize=20)
axes[1].set_title("FC, th=25%", fontsize=20)
axes[2].set_title("FC, th=1%", fontsize=20)
axes[3].set_title("Scaffolds", fontsize=20)

# Set y-label only for the first subplot
axes[0].set_ylabel('Correlation', fontsize=20)

# Set common y-limit
for ax in axes:
    ax.set_ylim(-0.1, 1)  # Changed lower limit from 0 to -0.2
    ax.set_xlabel('')

# Set x-labels with proper tick positions
for ax in axes:
    ax.set_xticks([0, 1])  # Set tick positions explicitly before setting labels
    ax.set_xticklabels(["Intra-sub", "Inter-sub"], rotation=0, fontsize=20)

# Enhance y-tick visualization
# First, define consistent y-ticks across all subplots
yticks = np.arange(0, 1.1, 0.2)
for i, ax in enumerate(axes):
    ax.set_yticks(yticks)
    if i == 0:
        # Only show y-ticks and labels on the first subplot
        ax.tick_params(axis='y', labelsize=18)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
        # Add subtle grid lines for better readability
        #ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)
    else:
        # Remove y-tick labels from other subplots but keep the ticks
        ax.tick_params(axis='y', which='both', left=True, labelleft=False)
        # Add matching grid to other subplots
        #ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)

# Enhance violin plot appearance
for i, ax in enumerate(axes):
    # Add subtle background color
    #ax.set_facecolor('#fafafa')
    
    # Customize violin plot elements
    for collection in ax.collections:
        if hasattr(collection, 'set_edgecolor'):
            collection.set_edgecolor('white')
            collection.set_linewidth(0.5)
        if hasattr(collection, 'set_alpha'):
            # Make violins slightly more transparent
            if 'violin' in str(type(collection)).lower():
                collection.set_alpha(0.8)



# Add annotations for each plot with effect sizes (enhanced styling)
pairs = [("intra_subject_fc100", "inter_subject_fc100")]
annot = Annotator(axes[0], pairs=pairs, data=df_fc100, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(round(es_fc100, 2))])
annot.configure(fontsize=18, text_offset=6, line_height=0.05, 
                line_width=1.5, color='black')
annot.annotate()

pairs = [("intra_subject_fc25", "inter_subject_fc25")]
annot = Annotator(axes[1], pairs=pairs, data=df_fc25, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(round(es_fc25, 2))])
annot.configure(fontsize=18, text_offset=5, line_height=0.05,
                line_width=1.5, color='black')
annot.annotate()

pairs = [("intra_subject_fc1", "inter_subject_fc1")]
annot = Annotator(axes[2], pairs=pairs, data=df_fc1, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(round(es_fc1, 2))])
annot.configure(fontsize=18, text_offset=5, line_height=0.05,
                line_width=1.5, color='black')
annot.annotate()

pairs = [("intra_subject_sc", "inter_subject_sc")]
annot = Annotator(axes[3], pairs=pairs, data=df_sc, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(round(es_sc, 2))])
annot.configure(fontsize=18, text_offset=0.5, line_height=0.05,
                line_width=1.5, color='black')
annot.annotate()


sb.despine()

# Add a title to the whole figure
#plt.suptitle('Intra-subject vs Inter-subject Correlations', fontsize=24, y=1.05)

# Adjust spacing between subplots
plt.subplots_adjust(wspace=0.05, bottom=0.2)

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2B.png'), dpi=300, bbox_inches='tight')
plt.show()

## Panel C (currently with 100 subjects data)

In [ ]:
# LOAD CORRELATION FINGERPRINTING DATA ##

sc = np.load(scaffolds_folder + 'scaffold_correlations_whole_cortex.npy')
fc100 = np.load(fc_folder + 'fc_correlations_whole_th100_cortex.npy')
fc25 = np.load(fc_folder + 'fc_correlations_whole_th25_cortex.npy')
fc1 = np.load(fc_folder + 'fc_correlations_whole_th1_cortex.npy')

In [ ]:
data_dir = config.HCP_100 + '/aal89/'
scaffold_dir = 'scaffolds/whole/'
sc_aal89 = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')

data_dir = config.HCP_100 + '/schaefer100/'
scaffold_dir = 'scaffolds/whole/'
sc_sch100 = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')


data_dir = config.HCP_100 + '/shen268/'
scaffold_dir = 'scaffolds/whole/'
sc_shen268 = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')

data_dir = config.HCP_100 + '/glasser379/'
scaffold_dir = 'scaffolds/whole/'
sc_glass379 = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole_cortex.npy')  # 360 cortical regions

In [ ]:
intra_subject_aal89 = np.diag(sc_aal89)
inter_subject_aal89 = sc_aal89[~np.eye(sc_aal89.shape[0], dtype=bool)]

intra_subject_sch100 = np.diag(sc_sch100)
inter_subject_sch100 = sc_sch100[~np.eye(sc_sch100.shape[0], dtype=bool)]

intra_subject_shen268 = np.diag(sc_shen268)
inter_subject_shen268 = sc_shen268[~np.eye(sc_shen268.shape[0], dtype=bool)]

intra_subject_glass379 = np.diag(sc_glass379)
inter_subject_glass379 = sc_glass379[~np.eye(sc_glass379.shape[0], dtype=bool)]

data_aal89 = {
    'distribution': (
        ['intra_subject_aal89'] * len(intra_subject_aal89)
        + ['inter_subject_aal89'] * len(inter_subject_aal89)
    ),
    'value': np.concatenate([intra_subject_aal89, inter_subject_aal89])
}

data_sch100 = {
    'distribution': (
        ['intra_subject_sch100'] * len(intra_subject_sch100)
        + ['inter_subject_sch100'] * len(inter_subject_sch100)
    ),
    'value': np.concatenate([intra_subject_sch100, inter_subject_sch100])
}

data_shen268 = {
    'distribution': (
        ['intra_subject_shen268'] * len(intra_subject_shen268)
        + ['inter_subject_shen268'] * len(inter_subject_shen268)
    ),
    'value': np.concatenate([intra_subject_shen268, inter_subject_shen268])
}

data_glass379 = {
    'distribution': (
        ['intra_subject_glass379'] * len(intra_subject_glass379)
        + ['inter_subject_glass379'] * len(inter_subject_glass379)
    ),
    'value': np.concatenate([intra_subject_glass379, inter_subject_glass379])
}

df_aal89 = pd.DataFrame(data_aal89)
df_sch100 = pd.DataFrame(data_sch100)
df_shen268 = pd.DataFrame(data_shen268)
df_glass379 = pd.DataFrame(data_glass379)

es = [compute_effsize(df_aal89.loc[df_aal89['distribution'] == 'intra_subject_aal89', 'value'], df_aal89.loc[df_aal89['distribution'] == 'inter_subject_aal89', 'value'], paired=True),
      compute_effsize(df_sch100.loc[df_sch100['distribution'] == 'intra_subject_sch100', 'value'], df_sch100.loc[df_sch100['distribution'] == 'inter_subject_sch100', 'value'], paired=True),
      compute_effsize(df_shen268.loc[df_shen268['distribution'] == 'intra_subject_shen268', 'value'], df_shen268.loc[df_shen268['distribution'] == 'inter_subject_shen268', 'value'], paired=True),
      compute_effsize(df_glass379.loc[df_glass379['distribution'] == 'intra_subject_glass379', 'value'], df_glass379.loc[df_glass379['distribution'] == 'inter_subject_glass379', 'value'], paired=True)]

In [ ]:
# Color palettes for the plots
blue_palette = sb.color_palette("Blues", n_colors=2)[::-1]  # For FC plots
orange_palette = sb.color_palette("Oranges", n_colors=2)[::-1]  # For scaffold plot

palette = orange_palette

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.ravel()


sb.violinplot(x="distribution", y="value", data=df_aal89, ax=axes[0], split=True, inner=None, palette=palette)
sb.stripplot(x="distribution", y="value", data=df_aal89, ax=axes[0], alpha=0.6, size=3, jitter=0.002, palette=palette)

sb.violinplot(x="distribution", y="value", data=df_sch100, ax=axes[1], split=True, inner=None, palette=palette)
sb.stripplot(x="distribution", y="value", data=df_sch100, ax=axes[1], alpha=0.6, size=3, jitter=0.002, palette=palette)

sb.violinplot(x="distribution", y="value", data=df_shen268, ax=axes[2], split=True, inner=None, palette=palette)
sb.stripplot(x="distribution", y="value", data=df_shen268, ax=axes[2], alpha=0.6, size=3, jitter=0.002, palette=palette)

sb.violinplot(x="distribution", y="value", data=df_glass379, ax=axes[3], split=True, inner=None, palette=palette)
sb.stripplot(x="distribution", y="value", data=df_glass379, ax=axes[3], alpha=0.6, size=3, jitter=0.002, palette=palette)

offset = transforms.ScaledTranslation(0.97, 0, fig.dpi_scale_trans)
for collection in axes[0].collections[-2:-1]:
    collection.set_transform(collection.get_transform() + offset)
for collection in axes[1].collections[-2:-1]:
    collection.set_transform(collection.get_transform() + offset)
for collection in axes[2].collections[-2:-1]:
    collection.set_transform(collection.get_transform() + offset)
for collection in axes[3].collections[-2:-1]:
    collection.set_transform(collection.get_transform() + offset)
for collection in axes[0].collections[-1:]:
    collection.set_transform(collection.get_transform() - offset)
for collection in axes[1].collections[-1:]:
    collection.set_transform(collection.get_transform() - offset)
for collection in axes[2].collections[-1:]:
    collection.set_transform(collection.get_transform() - offset)
for collection in axes[3].collections[-1:]:
    collection.set_transform(collection.get_transform() - offset)

axes[0].set_title("AAL 89", fontsize=24)
axes[1].set_title("Schaefer 100", fontsize=24)
axes[2].set_title("Shen 268", fontsize=24)
axes[3].set_title("Glasser 360", fontsize=24)

axes[0].set_xlabel('')
axes[0].set_ylim(0, .7)
axes[1].set_xlabel('')
axes[1].set_ylim(0, .7)
axes[2].set_xlabel('')
axes[2].set_ylim(0, .5)
axes[3].set_xlabel('')
axes[3].set_ylim(0, .5)

axes[0].set_ylabel('Correlation', fontsize=20)
axes[1].set_ylabel('')
axes[2].set_ylabel('Correlation', fontsize=20)
axes[3].set_ylabel('')
axes[1].set_yticklabels([])
axes[3].set_yticklabels([])
#axes[0].set_xticklabels(["Intra-sub", "Inter-sub"], rotation=45, fontsize=14)
#axes[1].set_xticklabels(["Intra-sub", "Inter-sub"], rotation=45, fontsize=14)
axes[0].set_xticklabels([])
axes[1].set_xticklabels([])
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(["Intra-sub", "Inter-sub"], rotation=45, fontsize=20)
axes[3].set_xticks([0, 1])
axes[3].set_xticklabels(["Intra-sub", "Inter-sub"], rotation=45, fontsize=20)

axes[0].tick_params(axis='y', labelsize=18)
axes[2].tick_params(axis='y', labelsize=18)

sb.despine()

plt.tight_layout()

pairs = [("intra_subject_aal89", "inter_subject_aal89")]
annot = Annotator(axes[0],pairs=pairs,data=df_aal89, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(np.round(es[0],2))])
annot.configure(fontsize=18)
annot.annotate(line_offset_to_group=0.1)

pairs = [("intra_subject_sch100", "inter_subject_sch100")]
annot = Annotator(axes[1],pairs=pairs,data=df_sch100, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(np.round(es[1],2))])
annot.configure(fontsize=18)
annot.annotate()

pairs = [("intra_subject_shen268", "inter_subject_shen268")]
annot = Annotator(axes[2],pairs=pairs,data=df_shen268, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(np.round(es[2],2))])
annot.configure(fontsize=18)
annot.annotate()

pairs = [("intra_subject_glass379", "inter_subject_glass379")]
annot = Annotator(axes[3],pairs=pairs,data=df_glass379, x="distribution", y="value")
annot.set_custom_annotations(['ES = ' + str(np.round(es[3],2))])
annot.configure(fontsize=18)
annot.annotate()

plt.show()



In [ ]:
# Load FC data for different atlases to compare with scaffolds
data_dir_aal89 = config.HCP_100 + '/aal89/'
fc_aal89 = np.load(data_dir_aal89 + 'fc/whole/fc_correlations_whole_th1.npy')

data_dir_sch100 = config.HCP_100 + '/schaefer100/'
fc_sch100 = np.load(data_dir_sch100 + 'fc/whole/fc_correlations_whole_th1.npy')

data_dir_shen268 = config.HCP_100 + '/shen268/'
fc_shen268 = np.load(data_dir_shen268 + 'fc/whole/fc_correlations_whole_th1.npy')

data_dir_glass379 = config.HCP_100 + '/glasser379/'
fc_glass379 = np.load(data_dir_glass379 + 'fc/whole/fc_correlations_whole_th1_cortex.npy')  # 360 cortical regions

# Extract intra/inter subject correlations for FC
intra_subject_fc_aal89 = np.diag(fc_aal89)
inter_subject_fc_aal89 = fc_aal89[~np.eye(fc_aal89.shape[0], dtype=bool)]

intra_subject_fc_sch100 = np.diag(fc_sch100)
inter_subject_fc_sch100 = fc_sch100[~np.eye(fc_sch100.shape[0], dtype=bool)]

intra_subject_fc_shen268 = np.diag(fc_shen268)
inter_subject_fc_shen268 = fc_shen268[~np.eye(fc_shen268.shape[0], dtype=bool)]

intra_subject_fc_glass379 = np.diag(fc_glass379)
inter_subject_fc_glass379 = fc_glass379[~np.eye(fc_glass379.shape[0], dtype=bool)]

# Create combined dataframes for FC and scaffolds comparison
# AAL89
data_combined_aal89 = {
    'condition': (
        ['FC_intra'] * len(intra_subject_fc_aal89) + 
        ['FC_inter'] * len(inter_subject_fc_aal89) +
        ['SC_intra'] * len(intra_subject_aal89) + 
        ['SC_inter'] * len(inter_subject_aal89)
    ),
    'modality': (
        ['FC'] * (len(intra_subject_fc_aal89) + len(inter_subject_fc_aal89)) +
        ['Scaffolds'] * (len(intra_subject_aal89) + len(inter_subject_aal89))
    ),
    'type': (
        ['Intra-sub'] * len(intra_subject_fc_aal89) + 
        ['Inter-sub'] * len(inter_subject_fc_aal89) +
        ['Intra-sub'] * len(intra_subject_aal89) + 
        ['Inter-sub'] * len(inter_subject_aal89)
    ),
    'value': np.concatenate([intra_subject_fc_aal89, inter_subject_fc_aal89, 
                            intra_subject_aal89, inter_subject_aal89])
}

# Schaefer100
data_combined_sch100 = {
    'condition': (
        ['FC_intra'] * len(intra_subject_fc_sch100) + 
        ['FC_inter'] * len(inter_subject_fc_sch100) +
        ['SC_intra'] * len(intra_subject_sch100) + 
        ['SC_inter'] * len(inter_subject_sch100)
    ),
    'modality': (
        ['FC'] * (len(intra_subject_fc_sch100) + len(inter_subject_fc_sch100)) +
        ['Scaffolds'] * (len(intra_subject_sch100) + len(inter_subject_sch100))
    ),
    'type': (
        ['Intra-sub'] * len(intra_subject_fc_sch100) + 
        ['Inter-sub'] * len(inter_subject_fc_sch100) +
        ['Intra-sub'] * len(intra_subject_sch100) + 
        ['Inter-sub'] * len(inter_subject_sch100)
    ),
    'value': np.concatenate([intra_subject_fc_sch100, inter_subject_fc_sch100, 
                            intra_subject_sch100, inter_subject_sch100])
}

# Shen268
data_combined_shen268 = {
    'condition': (
        ['FC_intra'] * len(intra_subject_fc_shen268) + 
        ['FC_inter'] * len(inter_subject_fc_shen268) +
        ['SC_intra'] * len(intra_subject_shen268) + 
        ['SC_inter'] * len(inter_subject_shen268)
    ),
    'modality': (
        ['FC'] * (len(intra_subject_fc_shen268) + len(inter_subject_fc_shen268)) +
        ['Scaffolds'] * (len(intra_subject_shen268) + len(inter_subject_shen268))
    ),
    'type': (
        ['Intra-sub'] * len(intra_subject_fc_shen268) + 
        ['Inter-sub'] * len(inter_subject_fc_shen268) +
        ['Intra-sub'] * len(intra_subject_shen268) + 
        ['Inter-sub'] * len(inter_subject_shen268)
    ),
    'value': np.concatenate([intra_subject_fc_shen268, inter_subject_fc_shen268, 
                            intra_subject_shen268, inter_subject_shen268])
}

# Glasser379
data_combined_glass379 = {
    'condition': (
        ['FC_intra'] * len(intra_subject_fc_glass379) + 
        ['FC_inter'] * len(inter_subject_fc_glass379) +
        ['SC_intra'] * len(intra_subject_glass379) + 
        ['SC_inter'] * len(inter_subject_glass379)
    ),
    'modality': (
        ['FC'] * (len(intra_subject_fc_glass379) + len(inter_subject_fc_glass379)) +
        ['Scaffolds'] * (len(intra_subject_glass379) + len(inter_subject_glass379))
    ),
    'type': (
        ['Intra-sub'] * len(intra_subject_fc_glass379) + 
        ['Inter-sub'] * len(inter_subject_fc_glass379) +
        ['Intra-sub'] * len(intra_subject_glass379) + 
        ['Inter-sub'] * len(inter_subject_glass379)
    ),
    'value': np.concatenate([intra_subject_fc_glass379, inter_subject_fc_glass379, 
                            intra_subject_glass379, inter_subject_glass379])
}

# Convert to dataframes
df_combined_aal89 = pd.DataFrame(data_combined_aal89)
df_combined_sch100 = pd.DataFrame(data_combined_sch100)
df_combined_shen268 = pd.DataFrame(data_combined_shen268)
df_combined_glass379 = pd.DataFrame(data_combined_glass379)

In [ ]:
# Create 2x2 subplot layout with boxplots comparing FC and scaffolds
fig, axes = plt.subplots(2, 2, figsize=(12, 13))
axes = axes.ravel()

# Define color palettes
blue_palette = sb.color_palette("Blues", n_colors=2)[::-1]  # For FC data
orange_palette = sb.color_palette("Oranges", n_colors=2)[::-1]  # For scaffold data

# AAL89 plot
sb.boxplot(x="modality", y="value", hue="type", data=df_combined_aal89, ax=axes[0], 
           dodge=True, showfliers=False, linewidth=2, width=0.6)

# Schaefer100 plot
sb.boxplot(x="modality", y="value", hue="type", data=df_combined_sch100, ax=axes[1], 
           dodge=True, showfliers=False, linewidth=2, width=0.6)


# Shen268 plot
sb.boxplot(x="modality", y="value", hue="type", data=df_combined_shen268, ax=axes[2], 
           dodge=True, showfliers=False, linewidth=2, width=0.6)


# Glasser379 plot
sb.boxplot(x="modality", y="value", hue="type", data=df_combined_glass379, ax=axes[3], 
           dodge=True, showfliers=False, linewidth=2, width=0.6)


# Manually set colors for all subplots
for ax_idx, ax in enumerate(axes):
    # Color the boxplots
    for i, patch in enumerate(ax.patches[:4]):  # Only first 4 patches are boxplots
        if i == 0:  # FC-Intra
            patch.set_facecolor(blue_palette[0])
            patch.set_alpha(0.8)
        elif i == 1:  # FC-Inter
            patch.set_facecolor(orange_palette[0])
            patch.set_alpha(0.8)
        elif i == 2:  # Scaffolds-Intra
            patch.set_facecolor(blue_palette[1])
            patch.set_alpha(0.8)
        elif i == 3:  # Scaffolds-Inter
            patch.set_facecolor(orange_palette[1])
            patch.set_alpha(0.8)

# Calculate and add effect size annotations for each comparison
# For FC Intra vs Inter
fc_aal89_es = compute_effsize(
    df_combined_aal89[(df_combined_aal89['modality'] == 'FC') & (df_combined_aal89['type'] == 'Intra-sub')]['value'],
    df_combined_aal89[(df_combined_aal89['modality'] == 'FC') & (df_combined_aal89['type'] == 'Inter-sub')]['value'],
    paired=False
)
fc_sch100_es = compute_effsize(
    df_combined_sch100[(df_combined_sch100['modality'] == 'FC') & (df_combined_sch100['type'] == 'Intra-sub')]['value'],
    df_combined_sch100[(df_combined_sch100['modality'] == 'FC') & (df_combined_sch100['type'] == 'Inter-sub')]['value'],
    paired=False
)
fc_shen268_es = compute_effsize(
    df_combined_shen268[(df_combined_shen268['modality'] == 'FC') & (df_combined_shen268['type'] == 'Intra-sub')]['value'],
    df_combined_shen268[(df_combined_shen268['modality'] == 'FC') & (df_combined_shen268['type'] == 'Inter-sub')]['value'],
    paired=False
)
fc_glass379_es = compute_effsize(
    df_combined_glass379[(df_combined_glass379['modality'] == 'FC') & (df_combined_glass379['type'] == 'Intra-sub')]['value'],
    df_combined_glass379[(df_combined_glass379['modality'] == 'FC') & (df_combined_glass379['type'] == 'Inter-sub')]['value'],
    paired=False
)

# For Scaffold Intra vs Inter
sc_aal89_es = compute_effsize(
    df_combined_aal89[(df_combined_aal89['modality'] == 'Scaffolds') & (df_combined_aal89['type'] == 'Intra-sub')]['value'],
    df_combined_aal89[(df_combined_aal89['modality'] == 'Scaffolds') & (df_combined_aal89['type'] == 'Inter-sub')]['value'],
    paired=False
)
sc_sch100_es = compute_effsize(
    df_combined_sch100[(df_combined_sch100['modality'] == 'Scaffolds') & (df_combined_sch100['type'] == 'Intra-sub')]['value'],
    df_combined_sch100[(df_combined_sch100['modality'] == 'Scaffolds') & (df_combined_sch100['type'] == 'Inter-sub')]['value'],
    paired=False
)
sc_shen268_es = compute_effsize(
    df_combined_shen268[(df_combined_shen268['modality'] == 'Scaffolds') & (df_combined_shen268['type'] == 'Intra-sub')]['value'],
    df_combined_shen268[(df_combined_shen268['modality'] == 'Scaffolds') & (df_combined_shen268['type'] == 'Inter-sub')]['value'],
    paired=False
)
sc_glass379_es = compute_effsize(
    df_combined_glass379[(df_combined_glass379['modality'] == 'Scaffolds') & (df_combined_glass379['type'] == 'Intra-sub')]['value'],
    df_combined_glass379[(df_combined_glass379['modality'] == 'Scaffolds') & (df_combined_glass379['type'] == 'Inter-sub')]['value'],
    paired=False
)


# Add effect size annotations using statannotations.Annotator
# For AAL89 - FC comparison
pairs_fc = [(('FC', 'Intra-sub'), ('FC', 'Inter-sub'))]
annot_fc_aal89 = Annotator(axes[0], pairs=pairs_fc, data=df_combined_aal89, 
                          x='modality', y='value', hue='type')
annot_fc_aal89.set_custom_annotations([f'ES = {fc_aal89_es:.2f}'])
annot_fc_aal89.configure(fontsize=20, text_offset=0.01, line_height=0.02,
                        line_width=1.5, color='black')
annot_fc_aal89.annotate(line_offset_to_group=0.02)

# For AAL89 - Scaffold comparison
pairs_sc = [(('Scaffolds', 'Intra-sub'), ('Scaffolds', 'Inter-sub'))]
annot_sc_aal89 = Annotator(axes[0], pairs=pairs_sc, data=df_combined_aal89,
                          x='modality', y='value', hue='type')
annot_sc_aal89.set_custom_annotations([f'ES = {sc_aal89_es:.2f}'])
annot_sc_aal89.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                        line_width=1.5, color='black')
annot_sc_aal89.annotate(line_offset_to_group=0.02)

# For Schaefer100 - FC comparison
annot_fc_sch100 = Annotator(axes[1], pairs=pairs_fc, data=df_combined_sch100,
                           x='modality', y='value', hue='type')
annot_fc_sch100.set_custom_annotations([f'ES = {fc_sch100_es:.2f}'])
annot_fc_sch100.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                         line_width=1.5, color='black')
annot_fc_sch100.annotate(line_offset_to_group=0.02)

# For Schaefer100 - Scaffold comparison
annot_sc_sch100 = Annotator(axes[1], pairs=pairs_sc, data=df_combined_sch100,
                           x='modality', y='value', hue='type')
annot_sc_sch100.set_custom_annotations([f'ES = {sc_sch100_es:.2f}'])
annot_sc_sch100.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                         line_width=1.5, color='black')
annot_sc_sch100.annotate(line_offset_to_group=0.02)

# For Shen268 - FC comparison
annot_fc_shen268 = Annotator(axes[2], pairs=pairs_fc, data=df_combined_shen268,
                            x='modality', y='value', hue='type')
annot_fc_shen268.set_custom_annotations([f'ES = {fc_shen268_es:.2f}'])
annot_fc_shen268.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                          line_width=1.5, color='black')
annot_fc_shen268.annotate(line_offset_to_group=0.02)

# For Shen268 - Scaffold comparison
annot_sc_shen268 = Annotator(axes[2], pairs=pairs_sc, data=df_combined_shen268,
                            x='modality', y='value', hue='type')
annot_sc_shen268.set_custom_annotations([f'ES = {sc_shen268_es:.2f}'])
annot_sc_shen268.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                          line_width=1.5, color='black')
annot_sc_shen268.annotate(line_offset_to_group=0.02)

# For Glasser379 - FC comparison
annot_fc_glass379 = Annotator(axes[3], pairs=pairs_fc, data=df_combined_glass379,
                             x='modality', y='value', hue='type')
annot_fc_glass379.set_custom_annotations([f'ES = {fc_glass379_es:.2f}'])
annot_fc_glass379.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                           line_width=1.5, color='black')
annot_fc_glass379.annotate(line_offset_to_group=0.02)

# For Glasser379 - Scaffold comparison
annot_sc_glass379 = Annotator(axes[3], pairs=pairs_sc, data=df_combined_glass379,
                             x='modality', y='value', hue='type')
annot_sc_glass379.set_custom_annotations([f'ES = {sc_glass379_es:.2f}'])
annot_sc_glass379.configure(fontsize=20, text_offset=0.02, line_height=0.02,
                           line_width=1.5, color='black')
annot_sc_glass379.annotate(line_offset_to_group=0.02)

# Set titles and labels
axes[0].set_title("AAL 90", fontsize=28, pad=20)
axes[1].set_title("Schaefer 100", fontsize=28, pad=20)
axes[2].set_title("Shen 268", fontsize=28, pad=20)
axes[3].set_title("Glasser 360", fontsize=28, pad=20)

# Set y-labels
axes[0].set_ylabel('Correlation', fontsize=24)
axes[1].set_ylabel('')
axes[2].set_ylabel('Correlation', fontsize=24)
axes[3].set_ylabel('')


yticks = np.linspace(0, 0.8, 5)  # Generate 6 ticks from 0 to 1.0
    
# Apply same y-ticks to all subplots
axes[0].set_ylim(-0.1, 0.9)
axes[1].set_ylim(-0.1, 0.9)
axes[2].set_ylim(0, 0.8)
axes[3].set_ylim(0, 0.8)

for ax in axes:
    ax.set_yticks(yticks)
    ax.tick_params(axis='y', labelsize=20)


# Set x-labels for all subplots
for i, ax in enumerate(axes):
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["FC", "Scaffolds"], rotation=0, fontsize=24)
    ax.set_xlabel('')

# Remove legends
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    if hasattr(ax, 'legend_') and ax.legend_:
        ax.legend_.remove()


sb.despine()

# Increase space between subplots
plt.subplots_adjust(hspace=0.4)

# Make room for legend
plt.subplots_adjust(top=0.90)

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2C_boxplots.png'), dpi=300, bbox_inches='tight')
plt.show()

# Panel D - Success Rate across Window Sizes with Bootstrap (Random Window Selection) (currently using 1200 subjects)

In [ ]:
# Compute valid_subs: subjects with both FC and scaffold files across ALL widths
# Also filters out subjects with inconsistent (non-modal) window counts
widths_all = [1100,1000,900,800,700,600,500,400,300,200,100,50]

valid_subs = None
for width in widths_all:
    subs_w, n_win = get_valid_subs_for_width(width, sup_folder)
    subs_set = set(subs_w)
    if valid_subs is None:
        valid_subs = subs_set
    else:
        valid_subs = valid_subs & subs_set
    print(f'  Width {width}: {len(subs_w)} valid ({n_win} windows), running intersection: {len(valid_subs)}')

valid_subs = sorted(valid_subs)
n_subs = len(valid_subs)
print(f'\nFinal: {n_subs} subjects with consistent data across all {len(widths_all)} widths')

In [ ]:
# Computed by scripts/07_fingerprinting_windows_1200.py (window-wise correlations do not fit in memory here)
# Success rate across window sizes, different sessions: one random window per subject pair at each bootstrap
import pickle as pk
with open(config.RESULTS + '/windows_fingerprinting.pck', 'rb') as f:
    win = pk.load(f)
success_rate_fc, success_rate_sc = win['success_rate_fc_different'], win['success_rate_sc_different']


In [ ]:
# Load and plot success rate across window sizes with bootstrap confidence intervals
import pickle as pk
import os

# Load saved bootstrap results
results_dir = config.RESULTS
bootstrap_results_path = os.path.join(results_dir, 'panel_d_success_rate_bootstrap.pck')

if os.path.isfile(bootstrap_results_path):
    with open(bootstrap_results_path, 'rb') as f:
        saved_data = pk.load(f)
        success_rate_fc = saved_data['success_rate_fc']
        success_rate_sc = saved_data['success_rate_sc']
        print(f"Loaded bootstrap results from {bootstrap_results_path}")

# Extract data for plotting
x = list(success_rate_fc.keys())
y_fc = [success_rate_fc[w]['mean'] for w in x]
y_sc = [success_rate_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [success_rate_fc[w]['std'] for w in x]
error_sc = [success_rate_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC success rate
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds success rate
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
#plt.title('Success Rate for Different Window Widths (Bootstrap)', fontsize=24)
plt.xlabel('Window width', fontsize=24)
plt.ylabel('Success Rate (%)', fontsize=24)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=20, rotation=45)
plt.tick_params(axis='y', labelsize=20)

# Set y-axis limits (success rate is percentage, so 0-100)
plt.ylim(0, 110)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=22, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2G_success_rate_bootstrap.png'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Success Rate Results:")
print("Window Size | FC Mean ± Std (%) | SC Mean ± Std (%)")
print("-" * 55)
for w in sorted(x):
    fc_mean = success_rate_fc[w]['mean']
    fc_std = success_rate_fc[w]['std']
    sc_mean = success_rate_sc[w]['mean']
    sc_std = success_rate_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Success Rate Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)
for w in sorted(x):
    fc_sr1 = success_rate_fc[w]['sr1_mean']
    fc_sr2 = success_rate_fc[w]['sr2_mean']
    sc_sr1 = success_rate_sc[w]['sr1_mean']
    sc_sr2 = success_rate_sc[w]['sr2_mean']
    print(f"{w:10d} | {fc_sr1:.1f} | {fc_sr2:.1f} | {sc_sr1:.1f} | {sc_sr2:.1f}")


# Panel D

In [ ]:
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50]
data_dir = sup_folder

fc_correlations = {}
sc_correlations = {}

for width in widths:
    fc_correlations[width] = {}
    sc_correlations[width] = {}
    fc0 = []
    fc1 = []
    sc0 = []
    sc1 = []
    for sub in valid_subs:
        fcs = np.load(data_dir+f'fc/{width}/{sub}')
        sc_path = get_scaffold_filename(sub, data_dir+f'scaffolds/windows_cortex/{width}/freq/')
        scs = load_scaffold(sc_path, n_regions_fc=fcs.shape[-1])
        n = fcs.shape[0]
        fc0.append(fcs[:n//2])
        fc1.append(fcs[n//2:])
        sc0.append(scs[:n//2])
        sc1.append(scs[n//2:])

    n_regions = fcs.shape[-1]
    fc0 = np.array(fc0).reshape(-1, n_regions, n_regions)
    fc1 = np.array(fc1).reshape(-1, n_regions, n_regions)
    #threshold FC
    fc0 = np.array([threshold_matrix(f,1).flatten() for f in fc0])
    fc1 = np.array([threshold_matrix(f,1).flatten() for f in fc1])
    sc0 = np.array(sc0).reshape(-1, n_regions, n_regions)
    sc1 = np.array(sc1).reshape(-1, n_regions, n_regions)
    sc0 = np.array([s.flatten() for s in sc0])
    sc1 = np.array([s.flatten() for s in sc1])
    m = fc0.shape[0]
    n_winps = int(fc0.shape[0]/n_subs) #n_windows per subject
    #compute correlations fc and scaffolds
    corrs_fc = np.corrcoef(fc0,fc1)[m:,:m]
    corrs_sc = np.corrcoef(sc0,sc1)[m:,:m]

    fc_correlations[width]['in'] = {}
    fc_correlations[width]['out'] = {}
    sc_correlations[width]['in'] = {}
    sc_correlations[width]['out'] = {}

    for l, sub_name in enumerate(valid_subs):
        sub_name = sub_name[:6]
        fc_row = corrs_fc[l*n_winps:(l+1)*n_winps,:].copy()
        sc_row = corrs_sc[l*n_winps:(l+1)*n_winps,:].copy()
        in_corr_fc = fc_row[:,l*n_winps:(l+1)*n_winps].mean()
        in_corr_sc = sc_row[:,l*n_winps:(l+1)*n_winps].mean()
        out_corr_fc = np.delete(fc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1).mean()
        out_corr_sc = np.delete(sc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1).mean()

        fc_correlations[width]['in'][sub_name] = in_corr_fc
        fc_correlations[width]['out'][sub_name] = out_corr_fc
        sc_correlations[width]['in'][sub_name] = in_corr_sc
        sc_correlations[width]['out'][sub_name] = out_corr_sc


In [ ]:
#rearrange distances for freq_scaff

data = []

for win_size, value in fc_correlations.items():
    for group, subjects in value.items():
        for subject, mean_corr in subjects.items():
                data.append({'win_size': win_size, 'group': group, 'subject': subject, 'value': mean_corr})

df_fc = pd.DataFrame(data)

df_fc

In [ ]:
# Create a custom color palette with varying shades of blue
blue_palette = sb.color_palette("Blues", n_colors=len(df_fc['group'].unique()))[::-1]

fig, ax = plt.subplots(figsize=(12, 6))

# Create a barplot of the distributions with increased separation and custom colors
sb.boxplot(x='win_size', y='value', data=df_fc, hue='group', dodge=0.6, palette=blue_palette)

# Calculate effect sizes for each window size
effect_sizes_fc = {}
for win_size in df_fc['win_size'].unique():
    intra_data = df_fc[(df_fc['win_size'] == win_size) & (df_fc['group'] == 'in')]['value']
    inter_data = df_fc[(df_fc['win_size'] == win_size) & (df_fc['group'] == 'out')]['value']
    es = compute_effsize(intra_data, inter_data, paired=True)
    effect_sizes_fc[win_size] = es

# Add effect size annotations for selected window sizes (to avoid overcrowding)
selected_windows = [1100, 800, 500, 200, 50]  # Select a subset of windows
for i, win_size in enumerate(selected_windows):
    if win_size in effect_sizes_fc:
        # Find the position of this window size in the plot
        win_positions = sorted(df_fc['win_size'].unique(), reverse=True)
        win_idx = win_positions.index(win_size)
        
        # Get max y value for this window size for annotation placement
        max_y = df_fc[df_fc['win_size'] == win_size]['value'].max()
        
        # Add effect size text annotation
        ax.text(win_idx, max_y + 0.02, f'ES={effect_sizes_fc[win_size]:.2f}', 
               ha='center', va='bottom', fontsize=10, fontweight='bold')

# Set the y-axis limits and ticks
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
#ax.yaxis.set_ticks(np.arange(0.4, 0.9, 0.1))

# Title and axis labels
plt.title('Functional Connectivity', fontsize=24)
#plt.xlabel('Window size', fontsize=14)
plt.xlabel('')
plt.ylabel('Correlations', fontsize=20)

# Increase the size of x and y ticks
#ax.tick_params(axis='x', labelsize=12)  # Increase the size of x ticks
ax.set_xticks([])  # Hide x-axis labels rather than setting empty labels
ax.tick_params(axis='y', labelsize=18)  # Increase the size of y ticks

sb.despine()

handles, labels = ax.get_legend_handles_labels()
custom_labels = {'in': 'Intra-sub', 'out': 'Inter-sub'}
for i, label in enumerate(labels):
    labels[i] = custom_labels.get(label, label)
plt.legend(handles, labels, title='', title_fontsize=18, fontsize=18, loc='upper left')

# Show the plot
plt.show()

In [ ]:
#rearrange distances for freq_scaff

data = []

for win_size, value in sc_correlations.items():
    for group, subjects in value.items():
        for subject, mean_corr in subjects.items():
                data.append({'win_size': win_size, 'group': group, 'subject': subject, 'value': mean_corr})

df_sc = pd.DataFrame(data)

df_sc

In [ ]:
# Create a custom color palette with varying shades of orange
orange_palette = sb.color_palette("Oranges", n_colors=len(df_sc['group'].unique()))[::-1]

fig, ax = plt.subplots(figsize=(12, 6))

# Create a barplot of the distributions with increased separation and custom colors
sb.boxplot(x='win_size', y='value', data=df_sc, hue='group', dodge=0.6, palette=orange_palette)

# Calculate effect sizes for each window size
effect_sizes_sc = {}
for win_size in df_sc['win_size'].unique():
    intra_data = df_sc[(df_sc['win_size'] == win_size) & (df_sc['group'] == 'in')]['value']
    inter_data = df_sc[(df_sc['win_size'] == win_size) & (df_sc['group'] == 'out')]['value']
    es = compute_effsize(intra_data, inter_data, paired=True)
    effect_sizes_sc[win_size] = es

# Add effect size annotations for selected window sizes (to avoid overcrowding)
selected_windows = [1100, 800, 500, 200, 50]  # Select a subset of windows
for i, win_size in enumerate(selected_windows):
    if win_size in effect_sizes_sc:
        # Find the position of this window size in the plot
        win_positions = sorted(df_sc['win_size'].unique(), reverse=True)
        win_idx = win_positions.index(win_size)
        
        # Get max y value for this window size for annotation placement
        max_y = df_sc[df_sc['win_size'] == win_size]['value'].max()
        
        # Add effect size text annotation
        ax.text(win_idx, max_y + 0.02, f'ES={effect_sizes_sc[win_size]:.2f}', 
               ha='center', va='bottom', fontsize=10, fontweight='bold')

# Set the y-axis limits and ticks
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
#ax.yaxis.set_ticks(np.arange(0.7, 0.9, 0.1))

# Title and axis labels
plt.title('Scaffolds', fontsize=24)
plt.xlabel('Window size', fontsize=18)
plt.ylabel('Correlations', fontsize=18)

# Increase the size of x and y ticks
ax.tick_params(axis='x', labelsize=18, rotation=45)  # Increase the size of x ticks
ax.tick_params(axis='y', labelsize=18)  # Increase the size of y ticks

sb.despine()

handles, labels = ax.get_legend_handles_labels()
custom_labels = {'in': 'Intra-sub', 'out': 'Inter-sub'}
for i, label in enumerate(labels):
    labels[i] = custom_labels.get(label, label)
plt.legend(handles, labels, title='', title_fontsize=18, fontsize=18, loc='upper left')


# Show the plot
plt.show()


# Panel E, effect sizes

In [ ]:
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50]
data_dir = sup_folder

es_fc = {}
es_sc = {}

for width in widths:
    fc0 = []
    fc1 = []
    sc0 = []
    sc1 = []
    for sub in valid_subs:
        fcs = np.load(data_dir+f'fc/{width}/{sub}')
        sc_path = get_scaffold_filename(sub, data_dir+f'scaffolds/windows_cortex/{width}/freq/')
        scs = load_scaffold(sc_path, n_regions_fc=fcs.shape[-1])
        n = fcs.shape[0]
        fc0.append(fcs[:n//2])
        fc1.append(fcs[n//2:])
        sc0.append(scs[:n//2])
        sc1.append(scs[n//2:])
    n_regions = fcs.shape[-1]
    fc0 = np.array(fc0).reshape(-1, n_regions, n_regions)
    fc1 = np.array(fc1).reshape(-1, n_regions, n_regions)
    #threshold FC
    fc0 = np.array([threshold_matrix(f,1).flatten() for f in fc0])
    fc1 = np.array([threshold_matrix(f,1).flatten() for f in fc1])
    sc0 = np.array(sc0).reshape(-1, n_regions, n_regions)
    sc1 = np.array(sc1).reshape(-1, n_regions, n_regions)
    sc0 = np.array([s.flatten() for s in sc0])
    sc1 = np.array([s.flatten() for s in sc1])
    m = fc0.shape[0]
    n_winps = int(fc0.shape[0]/n_subs) #n_windows per subject
    #compute correlations fc and scaffolds
    corrs_fc = np.corrcoef(fc0,fc1)[m:,:m]
    corrs_sc = np.corrcoef(sc0,sc1)[m:,:m]
    
    fc_correlations_matrix = np.zeros((n_subs,31))
    sc_correlations_matrix = np.zeros((n_subs,31))

    for l, sub_name in enumerate(valid_subs):
        sub_name = sub_name[:6]
        fc_correlations_list = []
        sc_correlations_list = []
        fc_row = corrs_fc[l*n_winps:(l+1)*n_winps,:].copy()
        sc_row = corrs_sc[l*n_winps:(l+1)*n_winps,:].copy()
        
        in_corr_fc = fc_row[:,l*n_winps:(l+1)*n_winps].mean()
        in_corr_sc = sc_row[:,l*n_winps:(l+1)*n_winps].mean()
        
        n_comparison = len(fc_row[:,l*n_winps:(l+1)*n_winps])
        
        fc_correlations_list.append(in_corr_fc)
        sc_correlations_list.append(in_corr_sc)
        
        out_corr_dist_fc = np.delete(fc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1)
        out_corr_dist_sc = np.delete(sc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1)
        
        for n_trial in range(30):
            out_corr_fc = np.array(random.sample(list(out_corr_dist_fc.flatten()), n_comparison)).mean()
            out_corr_sc = np.array(random.sample(list(out_corr_dist_sc.flatten()), n_comparison)).mean()
            fc_correlations_list.append(out_corr_fc)
            sc_correlations_list.append(out_corr_sc)
        
        fc_correlations_matrix[l] = np.array(fc_correlations_list)
        sc_correlations_matrix[l] = np.array(sc_correlations_list)
        
    es_fc[width] = [compute_effsize(fc_correlations_matrix[:,0],fc_correlations_matrix[:,i]) for i in range(1,31)]
    es_sc[width] = [compute_effsize(sc_correlations_matrix[:,0],sc_correlations_matrix[:,i]) for i in range(1,31)]

In [ ]:
x = widths

es_fc1 = np.array(list(es_fc.values()))
es_sc = np.array(list(es_sc.values()))

y_sc = es_sc.mean(axis=1)
y_fc1 = es_fc1.mean(axis=1)

error_fc1 = es_fc1.std(axis=1)*3
error_sc = es_sc.std(axis=1)*3

In [ ]:
# Plot success rate across window sizes with bootstrap confidence intervals

# Extract data for plotting
#x = x
y_fc = y_fc1
#y_sc = [success_rate_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = error_fc1
#error_sc = [success_rate_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC success rate
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds success rate
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Effect size for Different Window Widths', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Effect size', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (success rate is percentage, so 0-100)
#plt.ylim(15, 110)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2G_effectsize_bootstrap.pdf'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Effect Size Results:")
print("Window Size | FC Mean ± Std | SC Mean ± Std")
print("-" * 55)
for w in sorted(x):
    fc_mean = success_rate_fc[w]['mean']
    fc_std = success_rate_fc[w]['std']
    sc_mean = success_rate_sc[w]['mean']
    sc_std = success_rate_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Effect Size Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)
for w in sorted(x):
    fc_sr1 = success_rate_fc[w]['sr1_mean']
    fc_sr2 = success_rate_fc[w]['sr2_mean']
    sc_sr1 = success_rate_sc[w]['sr1_mean']
    sc_sr2 = success_rate_sc[w]['sr2_mean']
    print(f"{w:10d} | {fc_sr1:.1f} | {fc_sr2:.1f} | {sc_sr1:.1f} | {sc_sr2:.1f}")

## Effect size same sessions

In [ ]:
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50]
data_dir = sup_folder

es_fc = {}
es_sc = {}

for width in widths:
    fc0 = []
    fc1 = []
    sc0 = []
    sc1 = []
    for sub in valid_subs:
        fcs = np.load(data_dir+f'fc/{width}/{sub}')
        sc_path = get_scaffold_filename(sub, data_dir+f'scaffolds/windows_cortex/{width}/freq/')
        scs = load_scaffold(sc_path, n_regions_fc=fcs.shape[-1])
        n = fcs.shape[0]
        fc0.append(fcs[:n//2])
        fc1.append(fcs[:n//2])
        sc0.append(scs[:n//2])
        sc1.append(scs[:n//2])
    n_regions = fcs.shape[-1]
    fc0 = np.array(fc0).reshape(-1, n_regions, n_regions)
    fc1 = np.array(fc1).reshape(-1, n_regions, n_regions)
    #threshold FC
    fc0 = np.array([threshold_matrix(f,1).flatten() for f in fc0])
    fc1 = np.array([threshold_matrix(f,1).flatten() for f in fc1])
    sc0 = np.array(sc0).reshape(-1, n_regions, n_regions)
    sc1 = np.array(sc1).reshape(-1, n_regions, n_regions)
    sc0 = np.array([s.flatten() for s in sc0])
    sc1 = np.array([s.flatten() for s in sc1])
    m = fc0.shape[0]
    n_winps = int(fc0.shape[0]/n_subs) #n_windows per subject
    #compute correlations fc and scaffolds
    corrs_fc = np.corrcoef(fc0,fc1)[m:,:m]
    corrs_sc = np.corrcoef(sc0,sc1)[m:,:m]
    
    fc_correlations_matrix = np.zeros((n_subs,31))
    sc_correlations_matrix = np.zeros((n_subs,31))

    for l, sub_name in enumerate(valid_subs):
        sub_name = sub_name[:6]
        fc_correlations_list = []
        sc_correlations_list = []
        fc_row = corrs_fc[l*n_winps:(l+1)*n_winps,:].copy()
        sc_row = corrs_sc[l*n_winps:(l+1)*n_winps,:].copy()
        
        in_corr_fc = fc_row[:,l*n_winps:(l+1)*n_winps].mean()
        in_corr_sc = sc_row[:,l*n_winps:(l+1)*n_winps].mean()
        
        n_comparison = len(fc_row[:,l*n_winps:(l+1)*n_winps])
        
        fc_correlations_list.append(in_corr_fc)
        sc_correlations_list.append(in_corr_sc)
        
        out_corr_dist_fc = np.delete(fc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1)
        out_corr_dist_sc = np.delete(sc_row,np.s_[l*n_winps:(l+1)*n_winps],axis=1)
        
        for n_trial in range(30):
            out_corr_fc = np.array(random.sample(list(out_corr_dist_fc.flatten()), n_comparison)).mean()
            out_corr_sc = np.array(random.sample(list(out_corr_dist_sc.flatten()), n_comparison)).mean()
            fc_correlations_list.append(out_corr_fc)
            sc_correlations_list.append(out_corr_sc)
        
        fc_correlations_matrix[l] = np.array(fc_correlations_list)
        sc_correlations_matrix[l] = np.array(sc_correlations_list)
        
    es_fc[width] = [compute_effsize(fc_correlations_matrix[:,0],fc_correlations_matrix[:,i]) for i in range(1,31)]
    es_sc[width] = [compute_effsize(sc_correlations_matrix[:,0],sc_correlations_matrix[:,i]) for i in range(1,31)]

In [ ]:
x = widths

es_fc1 = np.array(list(es_fc.values()))
es_sc = np.array(list(es_sc.values()))

y_sc = es_sc.mean(axis=1)
y_fc1 = es_fc1.mean(axis=1)

error_fc1 = es_fc1.std(axis=1)*3
error_sc = es_sc.std(axis=1)*3

In [ ]:
# Plot success rate across window sizes with bootstrap confidence intervals

# Extract data for plotting
#x = x
y_fc = y_fc1
#y_sc = [success_rate_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = error_fc1
#error_sc = [success_rate_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC success rate
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds success rate
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Effect size for Different Window Widths', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Effect size', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (success rate is percentage, so 0-100)
#plt.ylim(15, 110)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2G_effectsize_bootstrap.pdf'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Effect Size Results:")
print("Window Size | FC Mean ± Std | SC Mean ± Std")
print("-" * 55)
for w in sorted(x):
    fc_mean = success_rate_fc[w]['mean']
    fc_std = success_rate_fc[w]['std']
    sc_mean = success_rate_sc[w]['mean']
    sc_std = success_rate_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Effect Size Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)
for w in sorted(x):
    fc_sr1 = success_rate_fc[w]['sr1_mean']
    fc_sr2 = success_rate_fc[w]['sr2_mean']
    sc_sr1 = success_rate_sc[w]['sr1_mean']
    sc_sr2 = success_rate_sc[w]['sr2_mean']
    print(f"{w:10d} | {fc_sr1:.1f} | {fc_sr2:.1f} | {sc_sr1:.1f} | {sc_sr2:.1f}")

# Panel F - Identifiability across Window Sizes with Bootstrap

In [ ]:
# Compute identifiability across different window sizes using bootstrap methods
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50]
data_dir = sup_folder
n_bootstrap = 1000  # Number of bootstrap samples

# Storage for identifiability results
identifiability_fc = {}
identifiability_sc = {}

for width in widths:
    print(f'Processing window size: {width}')
    
    fc0 = []
    fc1 = []
    sc0 = []
    sc1 = []
    
    # Load data for this window size
    for sub in valid_subs:
        fcs = np.load(data_dir+f'fc/{width}/{sub}')
        sc_path = get_scaffold_filename(sub, data_dir+f'scaffolds/windows_cortex/{width}/freq/')
        scs = load_scaffold(sc_path, n_regions_fc=fcs.shape[-1])
        n = fcs.shape[0]
        fc0.append(fcs[:n//2])
        fc1.append(fcs[n//2:])
        sc0.append(scs[:n//2])
        sc1.append(scs[n//2:])
    
    n_regions = fcs.shape[-1]  # dynamic: 300 for both datasets
    fc0 = np.array(fc0).reshape(-1, n_regions, n_regions)
    fc1 = np.array(fc1).reshape(-1, n_regions, n_regions)
    
    # Threshold FC
    fc0 = np.array([threshold_matrix(f,1).flatten() for f in fc0])
    fc1 = np.array([threshold_matrix(f,1).flatten() for f in fc1])
    
    sc0 = np.array(sc0).reshape(-1, n_regions, n_regions)
    sc1 = np.array(sc1).reshape(-1, n_regions, n_regions)
    sc0 = np.array([s.flatten() for s in sc0])
    sc1 = np.array([s.flatten() for s in sc1])
    
    m = fc0.shape[0]
    n_winps = int(fc0.shape[0]/n_subs)  # n_windows per subject
    
    # Compute correlations fc and scaffolds
    corrs_fc = np.corrcoef(fc0,fc1)[m:,:m]
    corrs_sc = np.corrcoef(sc0,sc1)[m:,:m]
    
    # Create subject-level correlation matrices
    fc_subject_corrs = np.zeros((n_subs, n_subs))
    sc_subject_corrs = np.zeros((n_subs, n_subs))
    
    # Compute subject-level correlations (mean across windows)
    for i in range(n_subs):
        for j in range(n_subs):
            fc_corr_block = corrs_fc[i*n_winps:(i+1)*n_winps, j*n_winps:(j+1)*n_winps]
            sc_corr_block = corrs_sc[i*n_winps:(i+1)*n_winps, j*n_winps:(j+1)*n_winps]
            
            fc_subject_corrs[i, j] = fc_corr_block.mean()
            sc_subject_corrs[i, j] = sc_corr_block.mean()
    
    # Bootstrap sampling for identifiability
    fc_idiff_bootstrap = []
    sc_idiff_bootstrap = []
    
    for bootstrap_iter in range(n_bootstrap):
        # Sample subjects with replacement
        bootstrap_indices = np.random.choice(n_subs, size=n_subs, replace=True)
        
        # Create bootstrap correlation matrices
        fc_bootstrap = fc_subject_corrs[np.ix_(bootstrap_indices, bootstrap_indices)]
        sc_bootstrap = sc_subject_corrs[np.ix_(bootstrap_indices, bootstrap_indices)]
        
        # Compute identifiability for this bootstrap sample
        fc_idiff, _, _ = compute_idiff(fc_bootstrap, kind='corr')
        sc_idiff, _, _ = compute_idiff(sc_bootstrap, kind='corr')
        
        fc_idiff_bootstrap.append(fc_idiff)
        sc_idiff_bootstrap.append(sc_idiff)
    
    # Store results
    identifiability_fc[width] = {
        'mean': np.mean(fc_idiff_bootstrap),
        'std': np.std(fc_idiff_bootstrap),
        'bootstrap_samples': fc_idiff_bootstrap
    }
    
    identifiability_sc[width] = {
        'mean': np.mean(sc_idiff_bootstrap),
        'std': np.std(sc_idiff_bootstrap),
        'bootstrap_samples': sc_idiff_bootstrap
    }

print('Identifiability computation completed!')

In [ ]:
# Plot identifiability across window sizes with bootstrap confidence intervals

# Extract data for plotting
x = list(identifiability_fc.keys())
y_fc = [identifiability_fc[w]['mean'] for w in x]
y_sc = [identifiability_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [identifiability_fc[w]['std'] for w in x]
error_sc = [identifiability_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(10,8))

# Plot FC identifiability
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds identifiability
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Identifiability for Different Window Widths (Bootstrap)', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Identifiability (I_diff)', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Add grid for better readability
plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2F_identifiability_bootstrap.png'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Identifiability Results:")
print("Window Size | FC Mean ± Std | SC Mean ± Std")
print("-" * 50)
for w in sorted(x):
    fc_mean = identifiability_fc[w]['mean']
    fc_std = identifiability_fc[w]['std']
    sc_mean = identifiability_sc[w]['mean']
    sc_std = identifiability_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.3f} ± {fc_std:.3f} | {sc_mean:.3f} ± {sc_std:.3f}")

# Panel G - Success Rate across Window Sizes with Bootstrap (Random Window Selection)

In [ ]:
# Compute success rate across different window sizes using bootstrap methods
# For each bootstrap iteration, randomly select one window per subject
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50]
data_dir = sup_folder
n_bootstrap = 100  # Number of bootstrap samples

# Storage for success rate results
success_rate_fc = {}
success_rate_sc = {}

for width in widths:
    print(f'Processing window size: {width}')
    
    fc0 = []
    fc1 = []
    sc0 = []
    sc1 = []
    
    # Load data for this window size
    for sub in valid_subs:
        fcs = np.load(data_dir+f'fc/{width}/{sub}')
        sc_path = get_scaffold_filename(sub, data_dir+f'scaffolds/windows_cortex/{width}/freq/')
        scs = load_scaffold(sc_path, n_regions_fc=fcs.shape[-1])
        n = fcs.shape[0]
        fc0.append(fcs[:n//2])
        fc1.append(fcs[n//2:])
        sc0.append(scs[:n//2])
        sc1.append(scs[n//2:])
    
    n_regions = fcs.shape[-1]  # dynamic: 300 for both datasets
    fc0 = np.array(fc0).reshape(-1, n_regions, n_regions)
    fc1 = np.array(fc1).reshape(-1, n_regions, n_regions)
    
    # Threshold FC
    fc0 = np.array([threshold_matrix(f,1).flatten() for f in fc0])
    fc1 = np.array([threshold_matrix(f,1).flatten() for f in fc1])
    
    sc0 = np.array(sc0).reshape(-1, n_regions, n_regions)
    sc1 = np.array(sc1).reshape(-1, n_regions, n_regions)
    sc0 = np.array([s.flatten() for s in sc0])
    sc1 = np.array([s.flatten() for s in sc1])
    
    m = fc0.shape[0]
    n_winps = int(fc0.shape[0]/n_subs)  # n_windows per subject
    
    # Compute correlations fc and scaffolds
    corrs_fc = np.corrcoef(fc0,fc1)[m:,:m]
    corrs_sc = np.corrcoef(sc0,sc1)[m:,:m]
    
    # Bootstrap sampling for success rate with random window selection
    fc_success_rate1_bootstrap = []
    fc_success_rate2_bootstrap = []
    sc_success_rate1_bootstrap = []
    sc_success_rate2_bootstrap = []
    
    for bootstrap_iter in range(n_bootstrap):
        # Create similarity matrices by randomly selecting one window per subject
        fc_similarity_matrix = np.zeros((n_subs, n_subs))
        sc_similarity_matrix = np.zeros((n_subs, n_subs))
        
        # For each subject pair, randomly select one window from each subject
        for i in range(n_subs):
            for j in range(n_subs):
                # Randomly select one window from subject i and one from subject j
                win_i = np.random.randint(0, n_winps)
                win_j = np.random.randint(0, n_winps)
                
                # Get the correlation between these specific windows
                global_idx_i = i * n_winps + win_i
                global_idx_j = j * n_winps + win_j
                
                fc_similarity_matrix[i, j] = corrs_fc[global_idx_i, global_idx_j]
                sc_similarity_matrix[i, j] = corrs_sc[global_idx_i, global_idx_j]
        
        # Compute success rate for this bootstrap sample
        fc_sr1, fc_sr2 = compute_success_rate(fc_similarity_matrix, kind='corr')
        sc_sr1, sc_sr2 = compute_success_rate(sc_similarity_matrix, kind='corr')
        
        fc_success_rate1_bootstrap.append(fc_sr1)
        fc_success_rate2_bootstrap.append(fc_sr2)
        sc_success_rate1_bootstrap.append(sc_sr1)
        sc_success_rate2_bootstrap.append(sc_sr2)
    
    # Compute mean success rate (average of both directions)
    fc_success_rate_mean_bootstrap = [(sr1 + sr2) / 2 for sr1, sr2 in zip(fc_success_rate1_bootstrap, fc_success_rate2_bootstrap)]
    sc_success_rate_mean_bootstrap = [(sr1 + sr2) / 2 for sr1, sr2 in zip(sc_success_rate1_bootstrap, sc_success_rate2_bootstrap)]
    
    # Store results
    success_rate_fc[width] = {
        'mean': np.mean(fc_success_rate_mean_bootstrap),
        'std': np.std(fc_success_rate_mean_bootstrap),
        'bootstrap_samples': fc_success_rate_mean_bootstrap,
        'sr1_mean': np.mean(fc_success_rate1_bootstrap),
        'sr2_mean': np.mean(fc_success_rate2_bootstrap)
    }
    
    success_rate_sc[width] = {
        'mean': np.mean(sc_success_rate_mean_bootstrap),
        'std': np.std(sc_success_rate_mean_bootstrap),
        'bootstrap_samples': sc_success_rate_mean_bootstrap,
        'sr1_mean': np.mean(sc_success_rate1_bootstrap),
        'sr2_mean': np.mean(sc_success_rate2_bootstrap)
    }

print('Success rate computation completed!')

In [ ]:
# Plot success rate across window sizes with bootstrap confidence intervals

# Extract data for plotting
x = list(success_rate_fc.keys())
y_fc = [success_rate_fc[w]['mean'] for w in x]
y_sc = [success_rate_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [success_rate_fc[w]['std'] for w in x]
error_sc = [success_rate_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC success rate
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds success rate
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
#plt.title('Success Rate for Different Window Widths (Bootstrap)', fontsize=24)
plt.xlabel('Window width', fontsize=24)
plt.ylabel('Success Rate (%)', fontsize=24)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=20, rotation=45)
plt.tick_params(axis='y', labelsize=20)

# Set y-axis limits (success rate is percentage, so 0-100)
plt.ylim(0, 110)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=22, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(os.path.join(figure_folder, 'fig2G_success_rate_bootstrap.png'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Success Rate Results:")
print("Window Size | FC Mean ± Std (%) | SC Mean ± Std (%)")
print("-" * 55)
for w in sorted(x):
    fc_mean = success_rate_fc[w]['mean']
    fc_std = success_rate_fc[w]['std']
    sc_mean = success_rate_sc[w]['mean']
    sc_std = success_rate_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Success Rate Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)
for w in sorted(x):
    fc_sr1 = success_rate_fc[w]['sr1_mean']
    fc_sr2 = success_rate_fc[w]['sr2_mean']
    sc_sr1 = success_rate_sc[w]['sr1_mean']
    sc_sr2 = success_rate_sc[w]['sr2_mean']
    print(f"{w:10d} | {fc_sr1:.1f} | {fc_sr2:.1f} | {sc_sr1:.1f} | {sc_sr2:.1f}")

# Panel G2 - Success Rate across Window Sizes with Bootstrap (Random Window Selection)

In [ ]:
# Computed by scripts/07_fingerprinting_windows_1200.py (window-wise correlations do not fit in memory here)
import pickle as pk
with open(config.RESULTS + '/windows_fingerprinting.pck', 'rb') as f:
    win = pk.load(f)
success_rate_fc, success_rate_sc = win['success_rate_fc_same'], win['success_rate_sc_same']


In [ ]:
# Plot success rate across window sizes with bootstrap confidence intervals

# Extract data for plotting
x = list(success_rate_fc.keys())
y_fc = [success_rate_fc[w]['mean'] for w in x]
y_sc = [success_rate_sc[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [success_rate_fc[w]['std'] for w in x]
error_sc = [success_rate_sc[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC success rate
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds success rate
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Success Rate for Different Window Widths, same Session', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Success Rate (%)', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (success rate is percentage, so 0-100)
plt.ylim(0, 110)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(config.FIGURES_RAW + '/supplementary/success_rate_same_session.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\nSummary of Success Rate Results:")
print("Window Size | FC Mean ± Std (%) | SC Mean ± Std (%)")
print("-" * 55)
for w in sorted(x):
    fc_mean = success_rate_fc[w]['mean']
    fc_std = success_rate_fc[w]['std']
    sc_mean = success_rate_sc[w]['mean']
    sc_std = success_rate_sc[w]['std']
    print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Success Rate Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)
for w in sorted(x):
    fc_sr1 = success_rate_fc[w]['sr1_mean']
    fc_sr2 = success_rate_fc[w]['sr2_mean']
    sc_sr1 = success_rate_sc[w]['sr1_mean']
    sc_sr2 = success_rate_sc[w]['sr2_mean']
    print(f"{w:10d} | {fc_sr1:.1f} | {fc_sr2:.1f} | {sc_sr1:.1f} | {sc_sr2:.1f}")

# Panel H - Effect size across Window Sizes with Bootstrap (Random Window Selection), different sessions

In [ ]:
# Computed by scripts/07_fingerprinting_windows_1200.py (window-wise correlations do not fit in memory here)
import pickle as pk
with open(config.RESULTS + '/windows_fingerprinting.pck', 'rb') as f:
    win = pk.load(f)
effect_size_fc_different_sessions, effect_size_sc_different_sessions = win['effect_size_fc_different'], win['effect_size_sc_different']


In [ ]:
# Plot effect size across window sizes with bootstrap confidence intervals

# Extract data for plotting
x = list(effect_size_fc_different_sessions.keys())
y_fc = [effect_size_fc_different_sessions[w]['mean'] for w in x]
y_sc = [effect_size_sc_different_sessions[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [effect_size_fc_different_sessions[w]['std'] for w in x]
error_sc = [effect_size_sc_different_sessions[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC effect size
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds effect size
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Effect Size for Different Window Widths, different sessions', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Effect Size', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (effect size is not constrained, so 0-1)
#plt.ylim(0, 1)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(config.FIGURES_RAW + '/supplementary/effect_size_different_sessions.pdf', dpi=300, bbox_inches='tight')
plt.show()

# # Print summary statistics
# print("\nSummary of Effect Size Results:")
# print("Window Size | FC Mean ± Std (%) | SC Mean ± Std (%)")
# print("-" * 55)
# for w in sorted(x):
#     fc_mean = effect_size_fc[w]['mean']
#     fc_std = effect_size_fc[w]['std']
#     sc_mean = effect_size_sc[w]['mean']
#     sc_std = effect_size_sc[w]['std']
#     print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Effect Size Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)

In [ ]:
plt.plot(x, np.array(y_sc) / np.array(y_fc))

# Panel H - Effect size across Window Sizes with Bootstrap (Random Window Selection), same sessions

In [ ]:
# Computed by scripts/07_fingerprinting_windows_1200.py (window-wise correlations do not fit in memory here)
import pickle as pk
with open(config.RESULTS + '/windows_fingerprinting.pck', 'rb') as f:
    win = pk.load(f)
effect_size_fc_same_sessions, effect_size_sc_same_sessions = win['effect_size_fc_same'], win['effect_size_sc_same']


In [ ]:
# Plot effect size across window sizes with bootstrap confidence intervals

# Extract data for plotting
x = list(effect_size_fc_same_sessions.keys())
y_fc = [effect_size_fc_same_sessions[w]['mean'] for w in x]
y_sc = [effect_size_sc_same_sessions[w]['mean'] for w in x]

# Standard deviations (used as confidence intervals)
error_fc = [effect_size_fc_same_sessions[w]['std'] for w in x]
error_sc = [effect_size_sc_same_sessions[w]['std'] for w in x]

# Create the plot
plt.figure(figsize=(12,8))

# Plot FC effect size
plt.plot(x, y_fc, label='Functional Connectivity', color='cornflowerblue', linewidth=2)
plt.scatter(x, y_fc, color='cornflowerblue', s=50, zorder=5)
plt.fill_between(x, np.array(y_fc) - np.array(error_fc), 
                 np.array(y_fc) + np.array(error_fc), 
                 alpha=0.3, color='cornflowerblue', label='FC ± 1 std')

# Plot Scaffolds effect size
plt.plot(x, y_sc, label='Scaffolds', color='sandybrown', linewidth=2)
plt.scatter(x, y_sc, color='sandybrown', s=50, zorder=5)
plt.fill_between(x, np.array(y_sc) - np.array(error_sc), 
                 np.array(y_sc) + np.array(error_sc), 
                 alpha=0.3, color='sandybrown', label='Scaffolds ± 1 std')

# Formatting
plt.title('Effect Size for Different Window Widths, same sessions', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Effect Size', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (effect size is not constrained, so 0-1)
#plt.ylim(0, 1)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(config.FIGURES_RAW + '/supplementary/effect_size_same_session.pdf', dpi=300, bbox_inches='tight')
plt.show()

# # Print summary statistics
# print("\nSummary of Effect Size Results:")
# print("Window Size | FC Mean ± Std (%) | SC Mean ± Std (%)")
# print("-" * 55)
# for w in sorted(x):
#     fc_mean = effect_size_fc[w]['mean']
#     fc_std = effect_size_fc[w]['std']
#     sc_mean = effect_size_sc[w]['mean']
#     sc_std = effect_size_sc[w]['std']
#     print(f"{w:10d} | {fc_mean:.1f} ± {fc_std:.1f} | {sc_mean:.1f} ± {sc_std:.1f}")

# Additional detailed statistics
print("\nDetailed Effect Size Statistics (SR1 vs SR2):")
print("Window Size | FC SR1 | FC SR2 | SC SR1 | SC SR2")
print("-" * 50)

In [ ]:
np.array(y_sc) / np.array(y_fc)

In [ ]:
plt.plot(x, np.array(y_sc) / np.array(y_fc))

In [ ]:
# Extract data for plotting
x = list(effect_size_fc_same_sessions.keys())
y_fc_same_session = [effect_size_fc_same_sessions[w]['mean'] for w in x]
y_sc_same_session = [effect_size_sc_same_sessions[w]['mean'] for w in x]

y_fc_different_session = [effect_size_fc_different_sessions[w]['mean'] for w in x]
y_sc_different_session = [effect_size_sc_different_sessions[w]['mean'] for w in x]

same_session_ratio = np.array(y_sc_same_session) / np.array(y_fc_same_session)
different_session_ratio = np.array(y_sc_different_session) / np.array(y_fc_different_session)

# Create the plot
plt.figure(figsize=(12,8))

# Plot ratio of scaffolds to functional connectivity for same sessions
plt.plot(x, same_session_ratio, label='Same session ratio', color='plum', linewidth=2)
plt.scatter(x, same_session_ratio, color='plum', s=50, zorder=5)

# Plot ratio of scaffolds to functional connectivity for different sessions
plt.plot(x, different_session_ratio, label='Different session ratio', color='green', linewidth=2)
plt.scatter(x, different_session_ratio, color='green', s=50, zorder=5)

# Formatting
plt.title('Scaffold to FC effect size ratio', fontsize=24)
plt.xlabel('Window width', fontsize=18)
plt.ylabel('Effect size ratio (scaffold / FC)', fontsize=18)

# Set x-axis ticks
plt.xticks([100,200,300,400,500,600,700,800,900,1000,1100])
plt.tick_params(axis='x', labelsize=18, rotation=45)
plt.tick_params(axis='y', labelsize=18)

# Set y-axis limits (effect size is not constrained, so 0-1)
#plt.ylim(0, 1)

# Add grid for better readability
#plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
plt.legend(fontsize=16, loc='best')

# Remove top and right spines
sb.despine()

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(config.FIGURES_RAW + '/supplementary/ratio.pdf', dpi=300, bbox_inches='tight')
plt.show()